# Attempt the **real ya||a** bridge on a Colab GPU

`pbg-yalla`'s `YallaProcess` drives the actual ya||a CUDA solver: it generates a
`.cu`, compiles it with `nvcc` against a ya||a checkout, and round-trips agent
state through the compiled binary each step. ya||a is GPU-only, so this needs an
NVIDIA GPU.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then run the cell below.

If the compile fails, the `❌` branch prints the full `nvcc` stderr — send that
back and the `.cu` template (`pbg_yalla/yalla_native.py`) can be patched. For a
CPU-only run, use `YallaReproductionProcess` instead.

In [ ]:
# === pbg-yalla: attempt the REAL ya||a bridge on a Colab GPU ===
import os, subprocess, sys

# 0. Confirm we actually have a GPU + nvcc
print(subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout)
assert subprocess.run(["which", "nvcc"], capture_output=True).returncode == 0, "no nvcc — pick a GPU runtime"

# 1. Minimal deps (skip the workspace/dashboard extras — the bridge only needs these)
subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                "numpy", "pyyaml", "process-bigraph", "bigraph-schema"], check=True)

# 2. Clone ya||a (the real simulator) and pbg-yalla
for url in ["https://github.com/germannp/yalla.git",
            "https://github.com/vivarium-collective/pbg-yalla.git"]:
    d = url.split("/")[-1][:-4]
    if not os.path.isdir(d):
        subprocess.run(["git", "clone", "--depth", "1", url], check=True)

os.environ["YALLA_HOME"] = os.path.abspath("yalla")
sys.path.insert(0, os.path.abspath("pbg-yalla"))

# 3. Match nvcc -arch to this GPU (e.g. "7.5" -> "sm_75")
cap = subprocess.run(["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip().splitlines()[0]
arch = "sm_" + cap.replace(".", "")
print("compiling for", arch)

# 4. Drive the REAL solver
from process_bigraph import allocate_core
from pbg_yalla.processes import YallaProcess
from pbg_yalla.yalla_native import YallaCudaUnavailable

core = allocate_core()
core.register_link("YallaProcess", YallaProcess)
proc = YallaProcess(config={
    "n_cells": 200, "force_kernel": "spring",
    "L_0": 0.5, "dt": 0.001, "seed": 1, "cuda_arch": arch,
}, core=core)

s0 = proc.initial_state()
print("initial :", {k: round(v, 4) if isinstance(v, float) else v for k, v in s0.items()})
try:
    for i in range(5):
        s = proc.update({}, interval=0.05)   # 50 take_step<spring> calls each
        print(f"step {i}:", {k: round(v, 4) if isinstance(v, float) else v for k, v in s.items()})
    print("\n✅ real ya||a ran. gyration_radius", round(s0["gyration_radius"], 4),
          "->", round(s["gyration_radius"], 4), "(spring should contract it)")
except YallaCudaUnavailable as e:
    print("\n❌ bridge could not run — likely an nvcc compile error below:\n")
    print(str(e))